# 01 — Data Ingestion & Data Handling
### Datasets: FaceForensics++ (C23) and Celeb-DF v2
This notebook handles:
- Kaggle API setup
- Downloading both datasets to Google Drive
- Component-based train/val/test splitting (FF++ only)
- MTCNN face crop extraction (16 frames per video)
- Saving index CSVs for all downstream notebooks

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install facenet-pytorch --quiet

print("Drive mounted and dependencies installed.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted and dependencies installed.


In [2]:
import os
import re
import gc
import cv2
import sys
import glob
import json
import random
import shutil
import numpy as np
import pandas as pd

from pathlib import Path
from collections import defaultdict, deque
from multiprocessing import Pool
from tqdm import tqdm

import torch
from facenet_pytorch import MTCNN

print("All libraries imported successfully.")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

All libraries imported successfully.
PyTorch version : 2.2.2+cu121
CUDA available  : True


In [3]:
# Project root and all subdirectory paths
PROJECT_ROOT = Path("/content/drive/MyDrive/deepfake_binary_project")

RAW_ROOT     = PROJECT_ROOT / "raw_datasets"
PROC_ROOT    = PROJECT_ROOT / "processed"
MODEL_DIR    = PROJECT_ROOT / "models"
INDEX_DIR    = PROJECT_ROOT / "index"

# Face crop directories for each dataset
FFPP_FACES_ROOT    = PROC_ROOT / "ffpp_face_crops_224"
CELEBDF_FACES_ROOT = PROC_ROOT / "celebdf_face_crops_224"

for p in [RAW_ROOT, PROC_ROOT, MODEL_DIR, INDEX_DIR,
          FFPP_FACES_ROOT, CELEBDF_FACES_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("Project directory structure initialized.")
print(f"  Root      : {PROJECT_ROOT}")
print(f"  Raw data  : {RAW_ROOT}")
print(f"  Processed : {PROC_ROOT}")
print(f"  Models    : {MODEL_DIR}")
print(f"  Index     : {INDEX_DIR}")

Project directory structure initialized.
  Root      : /content/drive/MyDrive/deepfake_binary_project
  Raw data  : /content/drive/MyDrive/deepfake_binary_project/raw_datasets
  Processed : /content/drive/MyDrive/deepfake_binary_project/processed
  Models    : /content/drive/MyDrive/deepfake_binary_project/models
  Index     : /content/drive/MyDrive/deepfake_binary_project/index


In [4]:
# Reproducibility and dataset configuration
SEED     = 42
IMG_SIZE = 224
NUM_FRAMES = 16
FACE_MARGIN = 0.30

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# FF++ class definitions and binary label mapping
VALID_CLASSES = ["original", "Deepfakes", "FaceSwap", "Face2Face", "NeuralTextures", "FaceShifter"]
FAKE_CLASSES  = [c for c in VALID_CLASSES if c != "original"]

BINARY_MAP        = {c: ("real" if c == "original" else "fake") for c in VALID_CLASSES}
BINARY_TARGET_MAP = {"real": 0, "fake": 1}

# Train / val / test split ratios
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device         : {device} ({torch.cuda.get_device_name(0)})")
print(f"Frames/video   : {NUM_FRAMES}")
print(f"Image size     : {IMG_SIZE}x{IMG_SIZE}")
print(f"Face margin    : {FACE_MARGIN}")
print(f"Split ratios   : train={TRAIN_RATIO} | val={VAL_RATIO} | test={TEST_RATIO}")

Device         : cuda (Tesla T4)
Frames/video   : 16
Image size     : 224x224
Face margin    : 0.3
Split ratios   : train=0.7 | val=0.15 | test=0.15


In [5]:
# Load Kaggle credentials from Drive (avoids re-uploading on every session)
kaggle_source = PROJECT_ROOT / "kaggle.json"

if not kaggle_source.exists():
    raise FileNotFoundError(f"kaggle.json not found at {kaggle_source}. Please upload it to your Drive project folder.")

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(str(kaggle_source), "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle API configured successfully.")

Kaggle API configured successfully.


In [6]:
FFPP_DIR = RAW_ROOT / "ff-c23"

def has_mp4s(folder):
    return len(glob.glob(os.path.join(str(folder), "**", "*.mp4"), recursive=True)) > 0

if has_mp4s(FFPP_DIR):
    print("FaceForensics++ already present on Drive — skipping download.")
else:
    print("Downloading FaceForensics++ (~16GB) ...")
    FFPP_DIR.mkdir(parents=True, exist_ok=True)
    os.system(f'kaggle datasets download -d xdxd003/ff-c23 -p "{FFPP_DIR}" --unzip')
    print("FaceForensics++ download complete.")

# Locate the dataset root folder
ff_candidates = list(RAW_ROOT.rglob("FaceForensics++_C23"))
if not ff_candidates:
    raise FileNotFoundError("FaceForensics++_C23 folder not found inside RAW_ROOT.")

FFPP_ROOT = ff_candidates[0]
print(f"FF++ root      : {FFPP_ROOT}")

FaceForensics++ already present on Drive — skipping download.
FF++ root      : /content/drive/MyDrive/deepfake_binary_project/raw_datasets/ff-c23/FaceForensics++_C23


In [7]:
# Validate all required class folders exist and report video counts
missing = [cls for cls in VALID_CLASSES if not (FFPP_ROOT / cls).exists()]
if missing:
    raise FileNotFoundError(f"Missing class folders: {missing}")

print("FF++ class folder verification:\n")
total = 0
for cls in VALID_CLASSES:
    count = len(list((FFPP_ROOT / cls).rglob("*.mp4")))
    label = BINARY_MAP[cls]
    print(f"  {cls:20s} [{label:4s}]  {count} videos")
    total += count

print(f"\n  Total videos : {total}")

FF++ class folder verification:

  original             [real]  1000 videos
  Deepfakes            [fake]  1000 videos
  FaceSwap             [fake]  1000 videos
  Face2Face            [fake]  1000 videos
  NeuralTextures       [fake]  1000 videos
  FaceShifter          [fake]  1000 videos

  Total videos : 6000


In [8]:
# Build a graph of video ID pairs to ensure source-target pairs
# are never split across train/val/test sets
pair_pat = re.compile(r"^(\d{3})_(\d{3})\.mp4$")

adj = defaultdict(set)
for cls in FAKE_CLASSES:
    for vid in (FFPP_ROOT / cls).rglob("*.mp4"):
        m = pair_pat.match(vid.name)
        if m:
            a, b = m.group(1), m.group(2)
            adj[a].add(b)
            adj[b].add(a)

# Extract connected components via BFS
visited, components = set(), []
for node in sorted(adj.keys()):
    if node in visited:
        continue
    q = deque([node])
    visited.add(node)
    comp = {node}
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in visited:
                visited.add(v)
                comp.add(v)
                q.append(v)
    components.append(comp)

# Append singleton original IDs not present in any pair
orig_ids = {p.stem for p in (FFPP_ROOT / "original").rglob("*.mp4")}
components += [{x} for x in sorted(orig_ids - set(adj.keys()))]

# Shuffle and split components
random.seed(SEED)
random.shuffle(components)

n           = len(components)
n_train     = int(n * TRAIN_RATIO)
n_val       = int(n * VAL_RATIO)

train_ids = set().union(*components[:n_train])
val_ids   = set().union(*components[n_train:n_train + n_val])
test_ids  = set().union(*components[n_train + n_val:])

# Sanity checks
assert len(train_ids & val_ids)  == 0, "Train/Val overlap detected."
assert len(train_ids & test_ids) == 0, "Train/Test overlap detected."
assert len(val_ids   & test_ids) == 0, "Val/Test overlap detected."

print(f"Total components : {n}")
print(f"Train IDs        : {len(train_ids)}")
print(f"Val IDs          : {len(val_ids)}")
print(f"Test IDs         : {len(test_ids)}")
print("\nNo leakage detected across splits.")

Total components : 500
Train IDs        : 700
Val IDs          : 150
Test IDs         : 150

No leakage detected across splits.


In [9]:
def get_split(filename):
    """Assign a split label to a video file based on its source/target IDs."""
    m = pair_pat.match(filename)
    if m:
        a, b = m.group(1), m.group(2)
        sa = "train" if a in train_ids else ("val" if a in val_ids else "test")
        sb = "train" if b in train_ids else ("val" if b in val_ids else "test")
        return sa if sa == sb else None
    stem = filename.replace(".mp4", "")
    if stem in train_ids: return "train"
    if stem in val_ids:   return "val"
    if stem in test_ids:  return "test"
    return None

rows, bad_files = [], []

for cls in VALID_CLASSES:
    for vid in (FFPP_ROOT / cls).rglob("*.mp4"):
        split = get_split(vid.name)
        if split is None:
            bad_files.append((cls, vid.name))
            continue
        rows.append({
            "path"         : str(vid),
            "file"         : vid.name,
            "source_class" : cls,
            "binary_label" : BINARY_MAP[cls],
            "binary_target": BINARY_TARGET_MAP[BINARY_MAP[cls]],
            "split"        : split,
            "dataset"      : "ffpp_c23"
        })

df_ffpp = pd.DataFrame(rows)

print(f"Total indexed videos : {len(df_ffpp)}")
print(f"Unassigned files     : {len(bad_files)}")
print(f"\nSplit & label distribution:\n")
print(df_ffpp.groupby(["split", "binary_label"]).size().unstack(fill_value=0))

Total indexed videos : 6000
Unassigned files     : 0

Split & label distribution:

binary_label  fake  real
split                   
test           750   150
train         3500   700
val            750   150


In [10]:
# Persist split DataFrames and summary statistics to Drive
train_df = df_ffpp[df_ffpp["split"] == "train"].reset_index(drop=True)
val_df   = df_ffpp[df_ffpp["split"] == "val"].reset_index(drop=True)
test_df  = df_ffpp[df_ffpp["split"] == "test"].reset_index(drop=True)

df_ffpp.to_csv(INDEX_DIR / "ffpp_full.csv",  index=False)
train_df.to_csv(INDEX_DIR / "ffpp_train.csv", index=False)
val_df.to_csv(INDEX_DIR  / "ffpp_val.csv",   index=False)
test_df.to_csv(INDEX_DIR / "ffpp_test.csv",   index=False)

split_stats = {
    "by_binary_label": df_ffpp.groupby(["split", "binary_label"]).size().unstack(fill_value=0).to_dict(),
    "by_source_class": df_ffpp.groupby(["split", "source_class"]).size().unstack(fill_value=0).to_dict()
}
with open(INDEX_DIR / "ffpp_split_stats.json", "w") as f:
    json.dump(split_stats, f, indent=2)

print("FF++ index files saved to Drive.")
print(f"  ffpp_full.csv   : {len(df_ffpp)} records")
print(f"  ffpp_train.csv  : {len(train_df)} records")
print(f"  ffpp_val.csv    : {len(val_df)} records")
print(f"  ffpp_test.csv   : {len(test_df)} records")
print(f"  ffpp_split_stats.json saved.")

FF++ index files saved to Drive.
  ffpp_full.csv   : 6000 records
  ffpp_train.csv  : 4200 records
  ffpp_val.csv    : 900 records
  ffpp_test.csv   : 900 records
  ffpp_split_stats.json saved.


In [11]:
CELEBDF_DIR = RAW_ROOT / "celeb-df-v2"

if has_mp4s(CELEBDF_DIR):
    print("Celeb-DF v2 already present on Drive — skipping download.")
else:
    print("Downloading Celeb-DF v2 ...")
    CELEBDF_DIR.mkdir(parents=True, exist_ok=True)
    os.system(f'kaggle datasets download -d reubensuju/celeb-df-v2 -p "{CELEBDF_DIR}" --unzip')
    print("Celeb-DF v2 download complete.")

print(f"\nCeleb-DF v2 root : {CELEBDF_DIR}")
print("\nFolder structure:")
for item in sorted(CELEBDF_DIR.iterdir()):
    count = len(list(item.rglob("*.mp4"))) if item.is_dir() else "-"
    print(f"  {item.name:25s}  {count} videos")

Celeb-DF v2 already present on Drive — skipping download.

Celeb-DF v2 root : /content/drive/MyDrive/deepfake_binary_project/raw_datasets/celeb-df-v2

Folder structure:
  Celeb-real                 590 videos
  Celeb-synthesis            5639 videos
  List_of_testing_videos.txt  - videos
  YouTube-real               300 videos


In [12]:
# Celeb-DF v2 is used exclusively as a cross-dataset test set
CELEBDF_FOLDERS = {
    "Celeb-real"     : "real",
    "YouTube-real"   : "real",
    "Celeb-synthesis": "fake",
}

celeb_rows = []
for folder, label in CELEBDF_FOLDERS.items():
    folder_path = CELEBDF_DIR / folder
    if not folder_path.exists():
        raise FileNotFoundError(f"Expected folder not found: {folder_path}")
    for vid in sorted(folder_path.rglob("*.mp4")):
        celeb_rows.append({
            "path"         : str(vid),
            "file"         : vid.name,
            "source_class" : folder,
            "binary_label" : label,
            "binary_target": BINARY_TARGET_MAP[label],
            "split"        : "test",
            "dataset"      : "celebdf_v2"
        })

df_celebdf = pd.DataFrame(celeb_rows)

print(f"Total Celeb-DF v2 videos : {len(df_celebdf)}")
print(f"\nBreakdown by source folder:\n")
print(df_celebdf.groupby(["source_class", "binary_label"]).size().reset_index(name="count").to_string(index=False))
print(f"\nReal  : {(df_celebdf.binary_target == 0).sum()}")
print(f"Fake  : {(df_celebdf.binary_target == 1).sum()}")

Total Celeb-DF v2 videos : 6529

Breakdown by source folder:

   source_class binary_label  count
     Celeb-real         real    590
Celeb-synthesis         fake   5639
   YouTube-real         real    300

Real  : 890
Fake  : 5639


In [13]:
df_celebdf.to_csv(INDEX_DIR / "celebdf_test.csv", index=False)

celeb_stats = {
    "total"         : len(df_celebdf),
    "real"          : int((df_celebdf.binary_target == 0).sum()),
    "fake"          : int((df_celebdf.binary_target == 1).sum()),
    "by_source_class": df_celebdf.groupby(["source_class", "binary_label"]).size().reset_index(name="count").to_dict(orient="records")
}
with open(INDEX_DIR / "celebdf_stats.json", "w") as f:
    json.dump(celeb_stats, f, indent=2)

print("Celeb-DF v2 index files saved to Drive.")
print(f"  celebdf_test.csv    : {len(df_celebdf)} records")
print(f"  celebdf_stats.json  saved.")

Celeb-DF v2 index files saved to Drive.
  celebdf_test.csv    : 6529 records
  celebdf_stats.json  saved.


## Data Handling
This section performs MTCNN-based face detection and extraction for both datasets.
For each video, 16 frames are sampled uniformly, the largest detected face is cropped
with a 30% margin, resized to 224×224, and saved as JPEG to Google Drive.
If no face is detected, a centre crop is used as fallback.

In [14]:
# MTCNN is used to detect and crop the largest face region in each frame.
# A margin is added around the bounding box to preserve facial context.
mtcnn = MTCNN(keep_all=False, device=device, post_process=False)

print("MTCNN face detector initialized.")
print(f"  Device      : {device}")
print(f"  Face margin : {FACE_MARGIN}")
print(f"  Output size : {IMG_SIZE}x{IMG_SIZE}")

MTCNN face detector initialized.
  Device      : cuda
  Face margin : 0.3
  Output size : 224x224


In [15]:
def extract_face_frames(video_path, out_dir, num_frames=NUM_FRAMES):
    """
    Extract and crop face regions from uniformly sampled video frames.
    Falls back to a centre crop if no face is detected in a given frame.
    """
    out_dir = Path(out_dir)

    # Skip if already fully extracted
    if out_dir.exists() and len(list(out_dir.glob("*.jpg"))) == num_frames:
        return

    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total <= 0:
        cap.release()
        return

    indices = np.linspace(0, total - 1, num_frames).astype(int)
    out_dir.mkdir(parents=True, exist_ok=True)

    for i, idx in enumerate(indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            # Save blank frame if read fails
            cv2.imwrite(str(out_dir / f"frame_{i:02d}.jpg"),
                        np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))
            continue

        rgb      = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        boxes, _ = mtcnn.detect(rgb)

        if boxes is not None and len(boxes) > 0:
            # Select largest detected face
            areas        = [(b[2] - b[0]) * (b[3] - b[1]) for b in boxes]
            x1, y1, x2, y2 = boxes[int(np.argmax(areas))].astype(int)
            w, h         = x2 - x1, y2 - y1
            mx, my       = int(w * FACE_MARGIN), int(h * FACE_MARGIN)
            x1 = max(0, x1 - mx);             y1 = max(0, y1 - my)
            x2 = min(rgb.shape[1], x2 + mx);  y2 = min(rgb.shape[0], y2 + my)
            crop = rgb[y1:y2, x1:x2]
        else:
            # Centre crop fallback
            H, W   = rgb.shape[:2]
            s      = min(H, W)
            y0, x0 = (H - s) // 2, (W - s) // 2
            crop   = rgb[y0:y0 + s, x0:x0 + s]

        crop = cv2.resize(crop, (IMG_SIZE, IMG_SIZE))
        cv2.imwrite(str(out_dir / f"frame_{i:02d}.jpg"),
                    cv2.cvtColor(crop, cv2.COLOR_RGB2BGR))

    cap.release()

print("Face extraction function defined.")
print(f"  Frames per video : {NUM_FRAMES}")
print(f"  Output size      : {IMG_SIZE}x{IMG_SIZE}")
print(f"  Face margin      : {FACE_MARGIN}")

Face extraction function defined.
  Frames per video : 16
  Output size      : 224x224
  Face margin      : 0.3


In [16]:
# Extract face crops for all FF++ videos in batches to manage memory efficiently
BATCH_SIZE_EXTRACT = 500
all_rows  = df_ffpp.to_dict("records")
total     = len(all_rows)
n_batches = (total + BATCH_SIZE_EXTRACT - 1) // BATCH_SIZE_EXTRACT

already_done = sum(
    1 for r in all_rows
    if (FFPP_FACES_ROOT / r["binary_label"] / Path(r["path"]).stem).exists()
    and len(list((FFPP_FACES_ROOT / r["binary_label"] /
                  Path(r["path"]).stem).glob("*.jpg"))) == NUM_FRAMES
)

print(f"FF++ face extraction status : {already_done}/{total} videos already processed.")

if already_done == total:
    print("All FF++ face crops present on Drive — skipping extraction.")
else:
    print(f"Extracting {total - already_done} remaining videos in {n_batches} batches...\n")

    for batch_idx in range(n_batches):
        start = batch_idx * BATCH_SIZE_EXTRACT
        end   = min(start + BATCH_SIZE_EXTRACT, total)
        batch = all_rows[start:end]

        needed = [
            row for row in batch
            if not (FFPP_FACES_ROOT / row["binary_label"] /
                    Path(row["path"]).stem).exists()
            or len(list((FFPP_FACES_ROOT / row["binary_label"] /
                         Path(row["path"]).stem).glob("*.jpg"))) != NUM_FRAMES
        ]

        if not needed:
            print(f"  Batch {batch_idx + 1}/{n_batches} — already complete, skipping.")
            continue

        print(f"  Batch {batch_idx + 1}/{n_batches} — processing {len(needed)} videos ...")
        for row in tqdm(needed, desc=f"Batch {batch_idx + 1}"):
            extract_face_frames(
                Path(row["path"]),
                FFPP_FACES_ROOT / row["binary_label"] / Path(row["path"]).stem
            )

        done_so_far = sum(
            1 for r in all_rows
            if (FFPP_FACES_ROOT / r["binary_label"] /
                Path(r["path"]).stem).exists()
            and len(list((FFPP_FACES_ROOT / r["binary_label"] /
                          Path(r["path"]).stem).glob("*.jpg"))) == NUM_FRAMES
        )
        print(f"  Progress : {done_so_far}/{total} complete.\n")

    print("FF++ face extraction complete.")

FF++ face extraction status : 0/6000 videos already processed.
Extracting 6000 remaining videos in 12 batches...

  Batch 1/12 — processing 500 videos ...


Batch 1: 100%|██████████| 500/500 [32:33<00:00,  3.91s/it]


  Progress : 500/6000 complete.

  Batch 2/12 — processing 500 videos ...


Batch 2: 100%|██████████| 500/500 [35:54<00:00,  4.31s/it]


  Progress : 1000/6000 complete.

  Batch 3/12 — processing 500 videos ...


Batch 3: 100%|██████████| 500/500 [32:14<00:00,  3.87s/it]


  Progress : 3500/6000 complete.

  Batch 4/12 — processing 500 videos ...


Batch 4: 100%|██████████| 500/500 [35:43<00:00,  4.29s/it]


  Progress : 6000/6000 complete.

  Batch 5/12 — already complete, skipping.
  Batch 6/12 — already complete, skipping.
  Batch 7/12 — already complete, skipping.
  Batch 8/12 — already complete, skipping.
  Batch 9/12 — already complete, skipping.
  Batch 10/12 — already complete, skipping.
  Batch 11/12 — already complete, skipping.
  Batch 12/12 — already complete, skipping.
FF++ face extraction complete.


In [17]:
# Verify all expected face crop folders are present and complete
missing    = []
incomplete = []

for row in df_ffpp.to_dict("records"):
    folder    = FFPP_FACES_ROOT / row["binary_label"] / Path(row["path"]).stem
    jpg_count = len(list(folder.glob("*.jpg"))) if folder.exists() else 0

    if not folder.exists():
        missing.append(Path(row["path"]).stem)
    elif jpg_count != NUM_FRAMES:
        incomplete.append((Path(row["path"]).stem, jpg_count))

print("FF++ face crop verification:")
print(f"  Total videos     : {len(df_ffpp)}")
print(f"  Missing folders  : {len(missing)}")
print(f"  Incomplete folders: {len(incomplete)}")

if len(missing) == 0 and len(incomplete) == 0:
    print("\n  All 6000 FF++ face crops verified successfully.")
else:
    if missing:
        print(f"\n  First 5 missing   : {missing[:5]}")
    if incomplete:
        print(f"\n  First 5 incomplete: {incomplete[:5]}")

FF++ face crop verification:
  Total videos     : 6000
  Missing folders  : 0
  Incomplete folders: 0

  All 6000 FF++ face crops verified successfully.


In [18]:
# Extract face crops for all Celeb-DF v2 videos using the same pipeline
all_celeb_rows  = df_celebdf.to_dict("records")
total_celeb     = len(all_celeb_rows)
n_batches_celeb = (total_celeb + BATCH_SIZE_EXTRACT - 1) // BATCH_SIZE_EXTRACT

already_done_celeb = sum(
    1 for r in all_celeb_rows
    if (CELEBDF_FACES_ROOT / r["binary_label"] / Path(r["path"]).stem).exists()
    and len(list((CELEBDF_FACES_ROOT / r["binary_label"] /
                  Path(r["path"]).stem).glob("*.jpg"))) == NUM_FRAMES
)

print(f"Celeb-DF v2 face extraction status : {already_done_celeb}/{total_celeb} videos already processed.")

if already_done_celeb == total_celeb:
    print("All Celeb-DF v2 face crops present on Drive — skipping extraction.")
else:
    print(f"Extracting {total_celeb - already_done_celeb} remaining videos in {n_batches_celeb} batches...\n")

    for batch_idx in range(n_batches_celeb):
        start = batch_idx * BATCH_SIZE_EXTRACT
        end   = min(start + BATCH_SIZE_EXTRACT, total_celeb)
        batch = all_celeb_rows[start:end]

        needed = [
            row for row in batch
            if not (CELEBDF_FACES_ROOT / row["binary_label"] /
                    Path(row["path"]).stem).exists()
            or len(list((CELEBDF_FACES_ROOT / row["binary_label"] /
                         Path(row["path"]).stem).glob("*.jpg"))) != NUM_FRAMES
        ]

        if not needed:
            print(f"  Batch {batch_idx + 1}/{n_batches_celeb} — already complete, skipping.")
            continue

        print(f"  Batch {batch_idx + 1}/{n_batches_celeb} — processing {len(needed)} videos ...")
        for row in tqdm(needed, desc=f"Batch {batch_idx + 1}"):
            extract_face_frames(
                Path(row["path"]),
                CELEBDF_FACES_ROOT / row["binary_label"] / Path(row["path"]).stem
            )

        done_so_far = sum(
            1 for r in all_celeb_rows
            if (CELEBDF_FACES_ROOT / r["binary_label"] /
                Path(r["path"]).stem).exists()
            and len(list((CELEBDF_FACES_ROOT / r["binary_label"] /
                          Path(r["path"]).stem).glob("*.jpg"))) == NUM_FRAMES
        )
        print(f"  Progress : {done_so_far}/{total_celeb} complete.\n")

    print("Celeb-DF v2 face extraction complete.")

Celeb-DF v2 face extraction status : 0/6529 videos already processed.
Extracting 6529 remaining videos in 14 batches...

  Batch 1/14 — processing 500 videos ...


Batch 1: 100%|██████████| 500/500 [37:02<00:00,  4.45s/it]


  Progress : 500/6529 complete.

  Batch 2/14 — processing 500 videos ...


Batch 2: 100%|██████████| 500/500 [38:32<00:00,  4.62s/it]


  Progress : 1000/6529 complete.

  Batch 3/14 — processing 500 videos ...


...

  Progress : 6000/6529 complete.

  Batch 13/14 — processing 500 videos ...


Batch 13: 100%|██████████| 500/500 [42:07<00:00,  5.06s/it]


  Progress : 6500/6529 complete.

  Batch 14/14 — processing 29 videos ...


Batch 14: 100%|██████████| 29/29 [02:27<00:00,  5.08s/it]


  Progress : 6529/6529 complete.

Celeb-DF v2 face extraction complete.


In [19]:
# Verify all expected face crop folders are present and complete
missing_celeb    = []
incomplete_celeb = []

for row in df_celebdf.to_dict("records"):
    folder    = CELEBDF_FACES_ROOT / row["binary_label"] / Path(row["path"]).stem
    jpg_count = len(list(folder.glob("*.jpg"))) if folder.exists() else 0

    if not folder.exists():
        missing_celeb.append(Path(row["path"]).stem)
    elif jpg_count != NUM_FRAMES:
        incomplete_celeb.append((Path(row["path"]).stem, jpg_count))

print("Celeb-DF v2 face crop verification:")
print(f"  Total videos      : {len(df_celebdf)}")
print(f"  Missing folders   : {len(missing_celeb)}")
print(f"  Incomplete folders: {len(incomplete_celeb)}")

if len(missing_celeb) == 0 and len(incomplete_celeb) == 0:
    print("\n  All 6529 Celeb-DF v2 face crops verified successfully.")
else:
    if missing_celeb:
        print(f"\n  First 5 missing   : {missing_celeb[:5]}")
    if incomplete_celeb:
        print(f"\n  First 5 incomplete: {incomplete_celeb[:5]}")

Celeb-DF v2 face crop verification:
  Total videos      : 6529
  Missing folders   : 0
  Incomplete folders: 0

  All 6529 Celeb-DF v2 face crops verified successfully.


In [20]:
# End-to-end summary of all ingestion and extraction outputs
print("=" * 60)
print("DATA INGESTION & HANDLING — COMPLETE SUMMARY")
print("=" * 60)

print("\nFaceForensics++ (C23)")
print(f"  Total videos     : {len(df_ffpp)}")
print(f"  Train            : {len(train_df)} (real: {(train_df.binary_target==0).sum()} | fake: {(train_df.binary_target==1).sum()})")
print(f"  Val              : {len(val_df)}   (real: {(val_df.binary_target==0).sum()} | fake: {(val_df.binary_target==1).sum()})")
print(f"  Test             : {len(test_df)}   (real: {(test_df.binary_target==0).sum()} | fake: {(test_df.binary_target==1).sum()})")
print(f"  Face crops root  : {FFPP_FACES_ROOT}")

print("\nCeleb-DF v2 (Cross-Dataset Test Set)")
print(f"  Total videos     : {len(df_celebdf)}")
print(f"  Real             : {(df_celebdf.binary_target==0).sum()}")
print(f"  Fake             : {(df_celebdf.binary_target==1).sum()}")
print(f"  Face crops root  : {CELEBDF_FACES_ROOT}")

print("\nIndex files saved to Drive:")
for f in sorted(INDEX_DIR.iterdir()):
    print(f"  {f.name}")

print("\nAll outputs verified and ready for downstream notebooks.")
print("=" * 60)

DATA INGESTION & HANDLING — COMPLETE SUMMARY

FaceForensics++ (C23)
  Total videos     : 6000
  Train            : 4200 (real: 700 | fake: 3500)
  Val              : 900   (real: 150 | fake: 750)
  Test             : 900   (real: 150 | fake: 750)
  Face crops root  : /content/drive/MyDrive/deepfake_binary_project/processed/ffpp_face_crops_224

Celeb-DF v2 (Cross-Dataset Test Set)
  Total videos     : 6529
  Real             : 890
  Fake             : 5639
  Face crops root  : /content/drive/MyDrive/deepfake_binary_project/processed/celebdf_face_crops_224

Index files saved to Drive:
  celebdf_stats.json
  celebdf_test.csv
  ffpp_full.csv
  ffpp_split_stats.json
  ffpp_test.csv
  ffpp_train.csv
  ffpp_val.csv

All outputs verified and ready for downstream notebooks.
